#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

#Reading from bronze layer

In [0]:
df = spark.read.table("workspace.bronze.crm_prd_info")

#Data transformations

## Trimming


In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name))) 

## Normalization

In [0]:
df = df.withColumn(
    'prd_line',
    F.when(col('prd_line') == 'M', 'Mountains')
     .when(col('prd_line') == 'R', 'Road')
     .when(col('prd_line') == 'S', 'Other Sales')
     .when(col('prd_line') == 'T', 'Touring')
     .otherwise('n/a')
)
    

##Product key parsing

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))


## Renaming columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)


##Date casting

In [0]:
df = df.withColumn('start_date', col('end_date').cast(DateType()))

In [0]:
df.display()